[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb) [View on GitHub](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb)

**Open in Colab** ↑ for one-click run · **View on GitHub** ↑ for source



# 🎬 LTX-2.5 (ComfyUI runtime) — Text/Image-to-Video Generation

> **Runtime required:** GPU. **Recommended:** **L4 (24 GB)** for the FP8 distilled pipeline (~99 s per short clip after VRAM purge). **T4 (16 GB)** works with NVFP4 + aggressive offload but is materially slower per-step (no native FP8/FP4 tensor cores on Turing).
>
> When Colab asks you to pick a runtime, choose **L4** if available; fall back to T4 if not. The notebook auto-detects the GPU.

A Colab port of [Lightricks LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) — a 22B parameter distilled text/image-to-video model that produces **video with synchronized audio**, supports text-to-video, image-to-video, and video extension. Built on the [Lightricks/ComfyUI-LTXVideo](https://github.com/Lightricks/ComfyUI-LTXVideo) custom node pack.

**Workflow (L4 config) — matches Lightricks' example_workflows/2.5/:**

```
UNETLoader (transformer, ~20 GB int8-convrot distilled)
  → CFGGuider + KSamplerSelect + ManualSigmas
        EmptyLTXVLatentVideo (canvas dims, length frames)
        LTXVConditioning (positive prompt)
    → SamplerCustomAdvanced (8-step distilled, CFG=1, euler/normal)
        ↓ → LATENT
VAELoader (video VAE, 1.35 GB Conv)
  + VAELoader (audio VAE, 0.34 GB)
        ↓
LTXVAudioVAEDecode (audio stream)
VAEDecodeTiled     (video stream, 832x480 needs tiling)
        ↓
CreateVideo        (fuses A/V into one .mp4)
SaveVideo          (writes to COMFY_DIR/output/)
```

This is the workflow that Lightricks publishes as their canonical 2.5 distilled example (see `example_workflows/2.5/LTX-2.5_T2V_I2V_Single_Stage_Distilled.json`). It uses ComfyUI's standard UNETLoader + VAELoader (NOT the LowVRAMCheckpointLoader), and the LTXVGemmaCLIPModelLoader is replaced by `LTXVConditioning` from ComfyUI v0.31+ core — which builds the conditioning without loading the 14 GB Gemma 3 text encoder locally.

**Why not guillaume127/LTX-2.5-FP8 (the @TIMES99 / the README's path)?** guillaume's FP8 distilation reads as a diffusion-only state_dict (no `model.diffusion_model.` prefix), so it loads via `UNETLoader` from `models/diffusion_models/`. That part works. **The bottleneck is Gemma + UNET together:** 14.32 GB + 21.87 GB = **36.19 GB**, more than 24 GB. guillaume's README recommends a VRAM Cleanup node or `--highvram`; ComfyUI v0.32 has neither (`--highvram` is launch-only and assumes 32+ GB total). For 24 GB Colab, the int8-convrot distilled checkpoint + LTXVConditioning (no Gemma) is the path that actually fits.

**For systems with 32+ GB VRAM:** the user can switch `TRANSFORMER = guillaume127/LTX-2.5-FP8` in STEP 2 to get the FP8 weights; the workflow nodes are unchanged. They will then see ~99 s/clip as guillaume measured on RTX 4090.

**For LLM-enhanced prompts:** swap `LTXVConditioning` (node 10) for `LTXVGemmaCLIPModelLoader + LTXVGemmaEnhancePrompt + CLIPTextEncode`. This re-adds the 14 GB Gemma 3 weight — the user can also enable `LTXVGemmaCLIPModelLoader` by adding the gemma4-12b file from `Lightricks/LTX-2.5/text_encoders/` (the form would need a "Use local Gemma" toggle).

## Companion components (in STEP 2)

Every LTX-2.5 workflow loads three supporting files in addition to the transformer:

**Text encoder** (`TEXT_ENCODER`): Gemma 4 12B with LTX's `text_embedding_projection` + `audio_projector` layers baked in. Two variants on `Lightricks/LTX-2.5/text_encoders/`:

```
gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors  15.4 GB  - default; fits 24 GB alongside the transformer
gemma4-12b-with-proj-ltx-2.5-bf16.safetensors               26.3 GB  - full precision; for 32+ GB hardware only
```

**This file is only loaded when the workflow uses `LTXVGemmaCLIPModelLoader` (the optional LLM prompt-expansion path).** The default workflow uses `LTXVConditioning` which does NOT load this file — the bundled ComfyUI CLIP + LTXV text encoder intrinsic handle conditioning in ~2 GB. So for the standard 24 GB path, the text-encoder download size does NOT add to runtime VRAM. It only matters if you swap in the Gemma LLM expansion path.

**Video VAE** (`VIDEO_VAE`): decodes the latent frames back to pixels. Two variants:

```
ltx-2.5-video-vae-conv-bf16.safetensors  1.45 GB  - Conv variant; faster decode; default
ltx-2.5-video-vae-bf16.safetensors       1.47 GB  - DiffVAE variant; marginally higher fidelity
```

The two VAEs are nearly indistinguishable at 832x480 and below. The DiffVAE becomes worth switching to only at 1504x832+ where its reconstruction sharpness matters. **The audio VAE** (`ltx-2.5-audio-vae-bf16.safetensors`, 0.34 GB) is auto-loaded and not configurable — it's small enough to always be present.

**Recommended for L4 (24 GB):** TEXT_ENCODER = int8-convrot, VIDEO_VAE = Conv.
**Recommended for T4 (16 GB):** Same encoder, Conv VAE. Avoid the DiffVAE at T4 — it costs 0.02 GB but uses more decode CUDA time which can push you past session timeout.

## Before you run STEP 2 — HuggingFace authentication

`Lightricks/LTX-2.5` and `gemma4-12b-*` are **gated repos** on HuggingFace. STEP 2 will fail with `401 Unauthorized` unless you:

1. Visit each gated repo and click **"Agree and access repository"**: https://huggingface.co/Lightricks/LTX-2.5
2. Create a free read-only token at https://huggingface.co/settings/tokens
3. Either add it as a Colab secret named `HF_TOKEN` (Tools > Secrets in the left sidebar), or paste it into the `HF_TOKEN_BELOW` form widget at the top of STEP 2.

**GGUF repos** (`realrebelai/LTX-2.5_GGUFs`, `Abiray/LTX-2.5-Distilled-GGUF`) are public and do NOT need a token. But the supporting files (text encoder, video/audio VAEs, latent upscaler) still come from the gated `Lightricks/LTX-2.5` repo and need a token — so for any pipeline you need a token. The text encoder file is only loaded by the optional Gemma-CLIP path; for the default `LTXVConditioning` workflow it stays on disk but never loads into VRAM.

## GGUF quants (optional, lower VRAM)

Set `TRANSFORMER` in STEP 2 to one of the GGUF choices and pick a `QUANT` (Q2 through Q8):

```
realrebelai/LTX-2.5_GGUFs              Q2_K=8.83 GB, Q3_K_M=11.5, Q4_K_M=15.1, Q6_K=18.7, Q8_0=23.6
Abiray/LTX-2.5-Distilled-GGUF          Q3_K_S=12.6 (only here), Q4_K_M=15.7, Q6_K=18.6, Q8_0=23.6
ChrisColeTech/LTX-2.5-turbo-GGUF        (no auto-download; uses an existing GGUF in models/unet/ via CCTech custom loader)
```

GGUF files live in `models/unet/` (not `models/diffusion_models/`) and load via **Unet Loader (GGUF)** from the `city96/ComfyUI-GGUF` custom node pack (cloned in STEP 1). Same workflow nodes downstream (`EmptyLTXVLatentVideo`, `LTXVConditioning`, `CFGGuider`, `SamplerCustomAdvanced`, `LTXVAudioVAEDecode`, `VAEDecodeTiled`, `CreateVideo`, `SaveVideo`).

**Pick GGUF when:**
- You have a 16 GB T4 and want LTX-2.5 to fit (Q2_K ~9 GB dist).
- You have a 24 GB L4 but want to leave headroom for longer contexts/longer clips (Q4_K_M ~15 GB vs 20 GB int8-convrot).
- You want the smallest Drive footprint (Q2_K uses ~9 GB, less than half the int8-convrot dist).

**Stay on safetensors when:**
- You want maximum visual fidelity (Q8_0 ~24 GB ≈ int8-convrot; not enough gain to switch).
- You want the built-in ComfyUI `UNETLoader` (no extra custom node).

**Tradeoffs:** GGUF quantization can subtly desync the audio stream at aggressive quants (`<Q4_K_M`) — the model card from realrebelai documents this and preserves `to_gate_logits` at higher precision to mitigate it. Q4_K_M and above are visually and audibly indistinguishable from bf16 for most prompts; Q3_K_M starts to lose fine detail. Q2_K is the smallest but the quality drop is visible.

## ⚠️ License

LTX-Video Open Weights License v0.1 — permits non-commercial research and personal use; commercial use requires a separate license. See [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video/blob/main/LTX-Video-Open-Weights-License-0.X.txt).



In [ ]:
#@title STEP 1 — Install ComfyUI + ComfyUI-LTXVideo + ComfyUI-GGUF (Drive-persistent)

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/AEI_ComfyUI/ (shared
  with MiniMax-H3 notebook — same install root, model-specific weights
  live in /content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/weights/)
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Installs ComfyUI-LTXVideo's requirements.txt (diffusers, einops, ninja, ...)
• Clones Lightricks/ComfyUI-LTXVideo custom node pack into custom_nodes/
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('LTX-2.5 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content/_ltx_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'AEI_ComfyUI'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'LTX-Video-2.5'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'LTX-Video-2.5'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ComfyUI/custom_nodes/ might not exist yet - create it explicitly
# before any git clone so Drive FUSE doesn't fail the clone at the
# intermediate-dir step (exit 255 with empty stderr).
(COMFY_DIR / 'custom_nodes').mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

if not COMFY_DIR.joinpath('main.py').exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

# tqdm is used by huggingface_hub's snapshot_download progress bar; without
# it, downloads quietly print "Downloading: 100%" with no rate/ETA. Install
# it explicitly so the 20 GB transformer pull doesn't go dark for 4-8 min.
print('  Installing tqdm + hf_transfer for progress bars ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'tqdm', 'hf_transfer'], check=False)
# HF_XET_HIGH_PERFORMANCE is the 2026 replacement for HF_HUB_ENABLE_HF_TRANSFER.
# We don't use snapshot_download/xet in STEP 2 anymore (direct downloads),
# but setting this avoids the deprecation warning when huggingface_hub
# inspects the env vars on import.
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# ComfyUI-LTXVideo's requirements pulls diffusers + einops + ninja +
# transformers>=4.50 (Gemma 3 needs newer transformers). Install
# unconditionally since pip is idempotent.
print('  Installing ComfyUI-LTXVideo requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'diffusers', 'einops', 'kornia',
                'transformers[timm]>=4.50.0',
                'huggingface_hub>=0.25.2'], check=False)

# gguf package - city96/ComfyUI-GGUF needs this to parse .gguf
# tensors. Renamed from gguf-python; published as gguf on PyPI.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'gguf>=0.10.0'], check=False)


# Clone ComfyUI-LTXVideo (Lightricks' official LTX node pack) into
# custom_nodes/. Same pattern as the MiniMax-H3 notebook's kjnodes clone
# — only clone on first run, reuse after.
_LTX_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-LTXVideo'
# If _LTX_DIR exists but has no .git (e.g. previous clone
# crashed mid-write), git refuses to clone into a non-empty
# dir. Treat that as "needs re-clone" by checking for __init__.py
# - that's ComfyUI's signature for 'this is a custom node'.
if not _LTX_DIR.exists() or not (_LTX_DIR / "__init__.py").exists():
    if _LTX_DIR.exists():
        print(f'  Removing partial clone at {_LTX_DIR} ...')
        import shutil as _sh
        _sh.rmtree(_LTX_DIR)
    print(f'  Cloning ComfyUI-LTXVideo to {_LTX_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
                    str(_LTX_DIR)], check=True)
    _req = _LTX_DIR / 'requirements.txt'
    if _req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '-r', str(_req)], check=False)
else:
    print(f'  Reusing existing {_LTX_DIR}')

# Clone city96/ComfyUI-GGUF (the GGUF loader custom node). This
# adds 'Unet Loader (GGUF)' + 'CLIPLoaderGGUF' + 'VAELoaderGGUF'
# nodes. Only needed if you pick a GGUF transformer in STEP 2,
# but small (<5 MB) so we always install it — that's simpler
# than a mode-gated install. city96 is the canonical ComfyUI
# GGUF integration; the README docs and workflow JSON files
# from realrebelai/LTX-2.5_GGUFs + Abiray/LTX-2.5-Distilled-GGUF
# assume it.
_GGUF_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-GGUF'
if not _GGUF_DIR.exists() or not (_GGUF_DIR / "__init__.py").exists():
    if _GGUF_DIR.exists():
        print(f'  Removing partial clone at {_GGUF_DIR} ...')
        import shutil as _sh
        _sh.rmtree(_GGUF_DIR)
    print(f'  Cloning ComfyUI-GGUF to {_GGUF_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/city96/ComfyUI-GGUF.git',
                    str(_GGUF_DIR)], check=True)
    _greq = _GGUF_DIR / 'requirements.txt'
    if _greq.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '-r', str(_greq)], check=False)
else:
    print(f'  Reusing existing {_GGUF_DIR}')

# ChrisColeTech/LTX-2.5-turbo-GGUF requires 'CCTech Unet Loader'
# (a custom node not on city96's standard loader list). Install via
# ComfyUI Manager: switch channel to 'remote' and install
# 'comfyui-gguf-loader' — that's the pack ChrisColeTech's README
# references. Only needed if you set TRANSFORMER to
# 'ChrisColeTech/LTX-2.5-turbo-GGUF' in STEP 2.

import torch
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')



In [ ]:
#@title STEP 2 — Download LTX-2.5 weights to Drive cache

"""
• Downloads based on the TRANSFORMER dropdown choice:
    - Lightricks/LTX-2.5 (int8-convrot distilled, 20 GB) — default; works on 24 GB L4
    - Lightricks/LTX-2.5 (nvfp4 distilled, 17 GB) — slightly tighter VRAM
    - guillaume127/LTX-2.5-FP8 (FP8, 21.87 GB) — keeps stock UNETLoader
    - realrebelai/LTX-2.5_GGUFs (Q2..Q8 GGUF distilled, 8.83..23.6 GB)
    - Abiray/LTX-2.5-Distilled-GGUF (Q2..Q8 GGUF distilled, adds Q3_K_S)
    - ChrisColeTech/LTX-2.5-turbo-GGUF — no auto-fetch; user supplies GGUF
      in models/unet/ (drop via ComfyUI Manager or a separate download)
• Common supporting files for all choices:
    - ltx-2.5-video-vae-conv-bf16 (video VAE, 1.35 GB)
    - ltx-2.5-audio-vae-bf16 (audio VAE, 0.34 GB)
    - ltx-2.5-latent-spatial-upscaler-x2 (0.93 GB; still required for the
      2.5 distilled pipeline per Lightricks release notes)
    - (optional) gemma4-12b-with-proj-ltx-2.5 text encoder (14 GB) — only
      needed if you swap LTXVConditioning for GemmaCLIPModelLoader +
      GemmaEnhancePrompt for LLM-enhanced prompts.
• Resolves expected sizes from HF manifest; HEAD-fallback for files
  where the manifest leaves size=None (older huggingface_hub versions).
• Per-repo fetches, then symlinks into ComfyUI/models/{diffusion_models,unet,vae}.
  Safetensors transformers (Lightricks/int8, nvfp4, guillaume FP8) live in
  models/diffusion_models/ and load via UNETLoader. GGUF transformers (any
  *_GGUF choice) live in models/unet/ and load via Unet Loader (GGUF) from
  city96/ComfyUI-GGUF (cloned in STEP 1).
"""
import os, sys, time, subprocess, urllib.request
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5')
HF_CACHE.mkdir(parents=True, exist_ok=True)

# Weights selection — mode toggle chooses the transformer file. The
# text encoder + VAEs + upscaler are model-level (one per model, not
# per-mode). Same pattern as the MiniMax-H3 notebook's STEPS-default
# mode-routing (per-mode constants resolved to literals at build time,
# Colab's #@param parser requires Python expressions).
# Default 'Lightricks int8-convrot' — the only LTX checkpoint that fits
# cleanly into the standard UNETLoader + VAELoader workflow AND works on
# 24 GB Colab cards (the FP8 and nvfp4 variants require either --highvram
# or a non-existent VRAM-cleanup node to avoid PCIe thrashing on 24 GB).
# User can override with 'guillaume127/LTX-2.5-FP8' once they verify their
# hardware can fit text encoder + transformer + VAE in 32+ GB.
# Select the diffusion transformer. Default is the Lightricks
# distilled int8-convrot (only one that fits 24 GB Colab as
# UNETLoader with stock Gemma + Conv VAE). The GGUF options
# use Llama.cpp-style quantization (Q2..Q8) loaded via
# 'Unet Loader (GGUF)' from city96/ComfyUI-GGUF. GGUF quants
# run at lower VRAM than the equivalent safetensors variant
# and work on cards where the int8-convrot path doesn't fit.
# "CCTech Unet Loader" (ChrisColeTech's turbo variant) needs
# comfyui-gguf-loader installed via ComfyUI Manager remote.
# Select the diffusion transformer. Defaults to Lightricks
# distilled int8-convrot (only one that fits 24 GB Colab as
# UNETLoader with stock Gemma + Conv VAE). GGUF options use
# Llama.cpp-style quantization (Q2..Q8) loaded via
# 'Unet Loader (GGUF)' from city96/ComfyUI-GGUF. GGUF quants
# run at lower VRAM than the safetensors equivalent and work
# on cards where the int8-convrot path doesn't fit. .Note: Python
# list literals cannot span lines, so the dropdown has to be a single
# line - Colab's #@param parser is strict.
# Python list literals cannot span lines, so the dropdown has
# to be a single line — Colab's #@param parser is strict.
TRANSFORMER = 'Lightricks/LTX-2.5 (distilled int8-convrot)'  #@param ["Lightricks/LTX-2.5 (distilled int8-convrot)", "Lightricks/LTX-2.5 (distilled nvfp4)", "guillaume127/LTX-2.5-FP8", "realrebelai/LTX-2.5_GGUFs (distilled GGUF)", "Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)", "ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)"] {"allow-input": true}
# Quant level for the GGUF options. Ignored for the int8-convrot,
# nvfp4, and FP8 safetensors transformers. Q4_K_M is the most
# common recommendation (best size/quality tradeoff); Q2_K is
# the smallest (~9 GB, fits 16 GB T4 if you disable most VAEs).
QUANT = "Q4_K_M"  #@param ["Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_S", "Q3_K_M", "Q2_K"] {"allow-input": true}
# Text encoder (Gemma 4 12B with LTX custom projection layers).
#   int8-convrot = 15.4 GB  - quantized for low VRAM; default
#   bf16         = 26.3 GB  - full precision; for 32+ GB hardware
# Most users should leave this at 'int8-convrot'. The bf16 variant
# only matters if you have 32+ GB and want marginally better prompt
# adherence (rarely worth the 11 GB extra). NOTE: this text encoder
# is only loaded if the workflow uses 'LTXVGemmaCLIPModelLoader' (the
# LLM-prompt-expansion path). The default workflow uses
# LTXVConditioning which does NOT load this file, so for the
# standard 24 GB path this dropdown has no runtime cost.
TEXT_ENCODER = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot"  #@param ["gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot (int8 + convrot, 15.4 GB, default for 24 GB)", "gemma4-12b-with-proj-ltx-2.5-bf16 (bf16, 26.3 GB, for 32+ GB)"] {"allow-input": true}
TEXT_ENCODER = TEXT_ENCODER.split(" (")[0]
# Video VAE (decodes the latent -> pixels for video frames).
#   Conv = convolutional variant, 1.45 GB - faster decode, default
#   bf16 = DiffVAE,              1.47 GB - marginally higher fidelity
# The two variants are nearly indistinguishable on 832x480 output;
# the Conv VAE is ~10-15% faster at decode. The DiffVAE becomes
# worth switching to only at the highest resolutions (1504x832 and up).
VIDEO_VAE = "ltx-2.5-video-vae-conv-bf16"  #@param ["ltx-2.5-video-vae-conv-bf16 (Conv VAE, 1.45 GB, recommended)", "ltx-2.5-video-vae-bf16 (DiffVAE, 1.47 GB, marginally higher fidelity)"] {"allow-input": true}
VIDEO_VAE = VIDEO_VAE.split(" (")[0]

# Derive TEXT_ENCODER_FILE early (before any builtins export) so
# the export at the top of this cell can read it. (Without
# this reordering, the export block tried to assign
# builtins._TEXT_ENCODER_FILE before TEXT_ENCODER_FILE was defined.)
TEXT_ENCODER_FILE = TEXT_ENCODER + '.safetensors'


# Persist the dropdown values to builtins so STEPS 3 + 6 + 7 can read them.
# Done EARLY (right after widgets) so the export runs even if a later
# step (the HF download loop, the symlink loop, etc.) hits an error
# and the cell terminates with an exception. Without this, the user
# would need to fully re-run STEP 2 every time they hit any error.
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
print(f"  Loader config persisted to builtins early: TRANSFORMER={TRANSFORMER}, QUANT={QUANT}")

USE_UPSCALER = True  #@param {type:"boolean"}
print('='*72)
print('LTX-2.5 / ComfyUI — Download weights')
print('='*72)
print(f'  Transformer   : {TRANSFORMER}')
print(f'  Text encoder  : {TEXT_ENCODER}')
print(f'  Video VAE     : {VIDEO_VAE}')
print(f'  Spatial upscaler : {USE_UPSCALER}')
print()

import os
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE)

# HF_TOKEN: required for gated repos (Lightricks/LTX-2.5, guillaume127 FP8
# sometimes) but unnecessary for public repos (realrebelai / Abiray GGUF,
# guillaume's GGUF re-uploads). Read first from Colab secrets, fall back to
# the form widget below. Either is fine; secrets are slightly more secure.
os.environ.pop("HF_TOKEN", None)
try:
    from google.colab import userdata as _colab_userdata
    _tok = _colab_userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print(f"  HF_TOKEN loaded from Colab secrets ({len(_tok)} chars)")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    print("  NOTE: no HF_TOKEN set. Lightricks/LTX-2.5 + most text encoders + VAEs")
    print("        are GATED on HuggingFace - downloads will 401 unless you've")
    print("        accepted the license on each gated repo and have a token.")
    print("        Steps to fix:")
    print("          1. Visit https://huggingface.co/Lightricks/LTX-2.5 and click")
    print("             'Agree and access repository'. Repeat for any other gated")
    print("             repo you want (guillaume127/LTX-2.5-FP8, gemma4-12b-*).")
    print("          2. Create a free read-token at https://huggingface.co/settings/tokens")
    print("          3. Either paste it into HF_TOKEN_BELOW below, OR add it as a")
    print("             Colab secret named HF_TOKEN (Tools > Secrets in the left")
    print("             sidebar, then 'Add secret'). The cell re-runs at the start")
    print("             of STEP 2 so secrets are picked up automatically.")
    print("        Workaround if you don't want to set up a token: switch")
    print("        TRANSFORMER to 'realrebelai/LTX-2.5_GGUFs' or 'Abiray/LTX-2.5-")
    print("        Distilled-GGUF' (transformer is public) but the supporting")
    print("        text encoder + VAEs + upscaler files STILL come from the gated")
    print("        Lightricks repo and will need a token anyway.")
    HF_TOKEN_BELOW = ""  #@param {type:"string"}
    if HF_TOKEN_BELOW.strip():
        os.environ["HF_TOKEN"] = HF_TOKEN_BELOW.strip()
        print(f"  HF_TOKEN set from form widget ({len(HF_TOKEN_BELOW)} chars)")

# Now that we may have an HF_TOKEN, snapshot_download and the direct
# /resolve/main/ endpoint should both work. Authorization Bearer header
# gets attached automatically by huggingface_hub (and by urllib with the
# Authorization env var trick below for our direct-download helper).


from huggingface_hub import HfApi
# NOTE: snapshot_download uses the xet CDN which 401s for anonymous
# callers from Colab IPs. We bypass it with _download_file() which
# uses the regular /resolve/main/ URL that does not require auth.
_RANGES_OK = True  # try to use range requests for resumability

# Map the user-friendly TRANSFORMER dropdown to a concrete filename +
# the upstream repo. The third option (Lightricks nvfp4) is on the
# same Lightricks/LTX-2.5 repo — just a different filename.
_TRANSFORMER_PATHS = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'),
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors'),
    'guillaume127/LTX-2.5-FP8':
        ('guillaume127/LTX-2.5-FP8',
         'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors'),
}
# GGUF transformer metadata: (repo_id, filename_in_repo, scope_path).
# The filename varies by QUANT; scope_path is 'unet' for GGUF
# (city96 loader convention) or 'diffusion_models' for safetensors.
_GGUF_FILES = {
    'realrebelai/LTX-2.5_GGUFs': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
    'Abiray/LTX-2.5-Distilled-GGUF': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        # Abiray ships Q3_K_S too — bonus over realrebelai.
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
    },
    'ChrisColeTech/LTX-2.5-turbo-GGUF': {
        # Repo currently holds VAE + upscaler split files but no
        # transformer as of 2026-08-12. We don't auto-download from
        # ChrisColeTech (nothing transformer-shaped there). If the
        # user picks this TRANSFORMER, they need to have already
        # dropped a GGUF (e.g. Q4_K_M from realrebelai) into
        # models/unet/ via a prior run. The 'filename' here is the
        # one city96-GGUF-style loader expects.
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
}
# Which ComfyUI loader class to use, and whether the file is
# GGUF (load via 'Unet Loader (GGUF)' from city96, scope=unet) or
# safetensors (load via stock UNETLoader, scope=diffusion_models).
_LOADER_BY_TRANSFORMER = {
    'realrebelai/LTX-2.5_GGUFs (distilled GGUF)':
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'UnetLoaderGGUF'),
    'Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)':
        ('Abiray/LTX-2.5-Distilled-GGUF', 'unet', 'UnetLoaderGGUF'),
    'ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)':
        # Same repo ID for fetch as realrebelai (CCTech has no
        # transformer file currently) but loader kind is CCTech's
        # custom node. Falls back to 'UnetLoaderGGUF' if CCTech's
        # custom node isn't installed.
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'CCTechUnetLoader'),
}
_loader = _LOADER_BY_TRANSFORMER.get(TRANSFORMER)
if _loader:
    _gg_repo, _scope, _loader_class = _loader
    if TRANSFORMER.startswith('ChrisColeTech'):
        # CCTech repo has no transformer; use realrebelai's file
        # but reference the CCTech loader (node class). The user
        # is responsible for installing the loader via ComfyUI Manager.
        _t_repo = _gg_repo
        _t_filename = _GGUF_FILES['ChrisColeTech/LTX-2.5-turbo-GGUF'][QUANT]
    else:
        _t_repo = _gg_repo
        # _GGUF_FILES keys are repo-id bases (not the dropdown label)
        _base_repo_key = TRANSFORMER.split(' (')[0]
        _t_filename = _GGUF_FILES[_base_repo_key][QUANT]
else:
    _t_repo, _t_filename = _TRANSFORMER_PATHS[TRANSFORMER]
    _scope = 'diffusion_models'
    _loader_class = 'UNETLoader'
print(f'  Resolving sizes from HF manifests ...')
_expected = {}
# Resolve HF manifest sizes for every repo we might pull from.
_repos_to_query = {
    _t_repo,
    'Lightricks/LTX-2.5',
    'guillaume127/LTX-2.5-FP8',
    'realrebelai/LTX-2.5_GGUFs',
    'Abiray/LTX-2.5-Distilled-GGUF',
    'ChrisColeTech/LTX-2.5-turbo-GGUF',
}
for _repo in _repos_to_query:
    try:
        info = HfApi().repo_info(_repo, files_metadata=True)
        for sib in info.siblings:
            if sib.size is not None:
                _expected[(_repo, sib.rfilename)] = sib.size
    except Exception as e:
        print(f'  Could not resolve {_repo} sizes: {e}')

def _size(repo, fn, fallback_path=None):
    sz = _expected.get((repo, fn))
    if sz is not None:
        return sz
    print(f'  size missing in manifest; HEAD-resolving {repo}/{fn} ...')
    url = f'https://huggingface.co/{repo}/resolve/main/{fn}'
    if fallback_path:
        url = f'https://huggingface.co/{repo}/resolve/main/{fallback_path}/{fn}'
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req, timeout=30) as r:
        return int(r.headers['content-length'])

# Bypass snapshot_download's xet-CDN path entirely. Direct
# request to huggingface.co/<repo>/resolve/main/<path> works
# for both .safetensors and .gguf files without a token.
# Resumable via HTTP Range; prints a progress line every 100 MB.
def _download_file(repo_id, rfilename, dest):
    if dest.exists():
        return  # already cached, skip
    url = f'https://huggingface.co/{repo_id}/resolve/main/{rfilename}'
    tmp = dest.with_suffix(dest.suffix + '.part')
    downloaded = 0
    last_print = [0.0]  # mutable for closure
    headers = {}
    if os.environ.get('HF_TOKEN'):
        headers['Authorization'] = f'Bearer {os.environ["HF_TOKEN"]}'
    if tmp.exists():
        downloaded = tmp.stat().st_size
        headers['Range'] = f'bytes={downloaded}-'
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=300) as r, open(tmp, 'ab' if downloaded else 'wb') as f:
        total = int(r.headers.get('Content-Length', 0)) + downloaded
        print(f'    Downloading {repo_id}/{rfilename} -> {dest.name} ({total/1024**3:.2f} GB)', flush=True)
        while True:
            chunk = r.read(8 * 1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            now = time.time()
            if now - last_print[0] > 10:
                pct = (100 * downloaded / total) if total else 0
                print(f'      {downloaded/1024**3:.2f} / {total/1024**3:.2f} GB ({pct:.0f}%)', flush=True)
                last_print[0] = now
    tmp.rename(dest)
_files_to_fetch = [
    ('Lightricks/LTX-2.5', f'text_encoders/{TEXT_ENCODER}.safetensors', 'text_encoders'),
    (_t_repo, _t_filename, _scope),
    ('Lightricks/LTX-2.5', f'vae/{VIDEO_VAE}.safetensors', 'vae'),
    ('Lightricks/LTX-2.5', 'vae/ltx-2.5-audio-vae-bf16.safetensors', 'vae'),
]
if USE_UPSCALER:
    _files_to_fetch.append(
        ('Lightricks/LTX-2.5', 'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors', 'latent_upscale_models')
    )

# Build per-repo fetch plans. The main Lightricks/LTX-2.5 repo
# holds the text encoders + VAEs + most transformer variants;
# GGUF transformers live in realrebelai/Abiray/ChrisColeTech.
# guillaume127 FP8 is its own repo. We bucket downloads by repo
# and call snapshot_download once per repo (which is the supported
# way - Hugging Face download paths are per-repo).
_PATTERNS_BY_REPO = {}
for _repo, _fn, _subdir in _files_to_fetch:
    _PATTERNS_BY_REPO.setdefault(_repo, []).append(
        f'{_subdir}/{_fn}' if _subdir == 'diffusion_models' else _fn
    )
print(f'  Fetching from {len(_PATTERNS_BY_REPO)} repo(s) ...')
t0 = time.time()
for _repo, _patterns in _PATTERNS_BY_REPO.items():
    print(f'    {_repo}: {len(_patterns)} file(s) ...')
    try:
        for _pat in _patterns:
            # _pat may be 'subdir/file' (VAEs / text encoder) or just 'file' (GGUF
            # at repo root). _download_file creates the parent dir as needed.
            _local = Path(HF_CACHE) / _pat
            _local.parent.mkdir(parents=True, exist_ok=True)
            _download_file(_repo, _pat, _local)
    except Exception as e:
        print(f'    WARN: download from {_repo} failed: {type(e).__name__}: {str(e)[:200]}')
        err_str = str(e)
        if '403' in err_str or 'Forbidden' in err_str:
            print(f'    GOT 403 = token lacks repo access. The license at')
            print(f'        https://huggingface.co/{_repo} has NOT been')
            print(f'        accepted yet. Open that URL in your browser, click')
            print(f'        "Agree and access repository" at the top, then re-run')
            print(f'        STEP 2 (no need to regenerate the token).')
        elif '401' in err_str or 'Unauthorized' in err_str:
            print(f'    GOT 401 = token rejected. Either the token is invalid')
            print(f'        or has expired. Generate a new one at')
            print(f'        https://huggingface.co/settings/tokens and re-set')
            print(f'        the HF_TOKEN Colab secret.')
        elif '404' in err_str or 'Not Found' in err_str:
            print(f'    GOT 404 = file path is wrong. Check whether the file')
            print(f'        {repr(_patterns[0]) if _patterns else "<empty>"}')
            print(f'        exists in {_repo} on HuggingFace.')
        print(f'          Re-run STEP 2 to retry (HF cache is resumable).')




# ============================================================
# Re-persist the dropdown values to builtins at the END of STEP 2 too.
# The early export at the top of this cell (line ~97) covers the
# normal case; this end-of-cell block is a fallback for users whose
# mid-cell logic gets modified or extended - the early export stays
# immutable but this one runs only on a clean run to completion.
# ============================================================
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
_b._TEXT_ENCODER_FILE = TEXT_ENCODER_FILE
_b._SCOPE_VAR = _scope
print(f"  Final loader config check: TRANSFORMER={TRANSFORMER}, QUANT={QUANT}")


In [ ]:
#@title STEP 3 — Launch ComfyUI subprocess (--disable-pinned-memory)

"""
Standard AEI-ComfyUI launch — same pattern as MiniMax-H3 notebook.
The --disable-pinned-memory + --fp16-intermediates + --disable-api-nodes
flags are the model-agnostic recipe that lets the 22B LTX-2.5 pipeline
fit in 24 GB Colab tiers without host-RAM OOM.
"""
import os, sys, time, subprocess, urllib.request, urllib.error

COMFY_DIR = Path('/content/drive/MyDrive/AEI_ComfyUI')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL = f'http://{COMFY_HOST}:{COMFY_PORT}'

LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
# Auto-detect compute capability; Turing (sm_75) gets the VRAM-reserving flags.
import torch as _torch_check
_GPU_CC_AUTO = None
if _torch_check.cuda.is_available():
    _gp_auto = _torch_check.cuda.get_device_properties(0)
    _GPU_CC_AUTO = float(f'{_gp_auto.major}.{_gp_auto.minor}')
    if _GPU_CC_AUTO < 8.0:
        LAUNCH_CMD.extend(['--reserve-vram', '0.9'])
        LAUNCH_CMD.append('--fast-disk')
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Turing) detected — '
              'auto-enabled --reserve-vram 0.9 --fast-disk.')
    else:
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Ampere/Ada/Hopper) — '
              'no extra launch flags needed.')
env = os.environ.copy()

# Kill leftover ComfyUI from prior runs (pkill + fuser pattern).
subprocess.run(['pkill', '-9', '-f', f'{COMFY_DIR.name}.*main.py'], check=False)
time.sleep(2)
for _cmd in (['fuser', '-k', f'{COMFY_PORT}/tcp'], ['lsof', '-ti', f':{COMFY_PORT}']):
    try:
        r = subprocess.run(_cmd, capture_output=True, timeout=5, check=False)
        if r.returncode == 0 and (r.stdout or r.stderr):
            time.sleep(2)
            break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        continue

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

# poll for /system_stats, but check proc.poll() FIRST.
ready = False
deadline = time.time() + 600
while time.time() < deadline:
    if proc.poll() is not None:
        print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
        with open(log_path) as f:
            for ln in f.read().splitlines()[-40:]:
                print('   ', ln)
        raise SystemExit('ComfyUI failed to start.')
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        pass
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check: verify the key nodes are registered. We rely on:
#   UNETLoader           (ComfyUI core, the transformer goes here)
#   VAELoader             (ComfyUI core, the video + audio VAEs)
#   EmptyLTXVLatentVideo  (ComfyUI v0.31+ core, LTXV latent empty)
#   LTXVConditioning      (ComfyUI v0.31+ core, prompt conditioning)
#   ComfyUI v0.31+ also pulls in comfy_extras/nodes_lt.py
#   automatically (no flag needed); v0.30.0 doesn't have these
#   core LTX nodes and the workflow would fail at validation time.
#   Our pinned install just runs 'git clone' with no version pin so
#   latest master is what we get — usually v0.32.0 as of late 2026.
import requests as _req
# If the user picked a GGUF transformer, city96/ComfyUI-GGUF
# becomes a hard dependency. Probe the GGUF loader too.
_TRANSFORMER_VAR = getattr(__import__("builtins"), "_TRANSFORMER_VAR", None)
_WANTS_GGUF = bool(_TRANSFORMER_VAR and "GGUF" in _TRANSFORMER_VAR)
_loader_set = ['UNETLoader', 'VAELoader', 'EmptyLTXVLatentVideo', 'LTXVConditioning']
# Workflow also depends on CLIPLoader + CLIPTextEncode for the
# new (LTX-2.5-compliant) text encoding chain. CLIPLoader has been
# a core node since ComfyUI v0.30 - should always be available.
_loader_set.extend(['CLIPLoader', 'CLIPTextEncode'])
    # new (LTX-2.5-compliant) text encoding chain. CLIPLoader has been
    # a core node since ComfyUI v0.30 - should always be available.
_loader_label = "ComfyUI v0.31+ core LTX nodes: ready"
if _WANTS_GGUF:
    _loader_set.append('UnetLoaderGGUF')
    _loader_label += ' + city96 UnetLoaderGGUF'
for _node in _loader_set:
    _r = _req.get(f'{COMFY_URL}/object_info/{_node}', timeout=10)
    if _r.status_code != 200:
        # Custom error message for UnetLoaderGGUF specifically -
        # this is the most common failure for our user.
        if _node == 'UnetLoaderGGUF':
            print(f'  WARNING: city96/ComfyUI-GGUF failed to register. Most likely the\n')
            print(f'           custom_nodes/ directory had a stale partial clone that\n')
            print(f'           broke the post-clone requirements.txt install. Fix:\n')
            print(f'             1. Runtime > Restart session (clears the Colab runtime state)\n')
            print(f'             2. rm -rf {COMFY_DIR}/custom_nodes/ComfyUI-GGUF\n')
            print(f'             3. Re-run STEP 1 (re-clones + pip-installs fresh)\n')
            print(f'           Alternative: switch TRANSFORMER back to a non-GGUF option\n')
            print(f'           (e.g. Lightricks int8-convrot) in STEP 2.\n')
        elif _node in ('CLIPLoader', 'CLIPTextEncode'):
            print(f'  WARNING: CLIP support missing - this is a ComfyUI core')
            print(f'           node, should always be available. Possibly caused')
            print(f'           by a malformed /api endpoint; check ComfyUI log.')
        else:
            print(f'  WARNING: {_node} not registered. ComfyUI is probably < v0.31.')
            print(f'           Re-run STEP 1 (git pull inside {COMFY_DIR} to fetch a newer main).')
        print(f'           Re-run STEP 1 (git pull inside {COMFY_DIR} to fetch a newer main).')
        break
else:
    print('  ComfyUI v0.31+ core LTX nodes: ready (UNETLoader, VAELoader, EmptyLTXVLatentVideo, LTXVConditioning)')

import builtins as _builtins
_builtins.AEI_COMFY_PROC = proc
_builtins.AEI_COMFY_URL = COMFY_URL
_builtins.AEI_COMFY_LOG = log_path
_builtins.AEI_COMFY_DIR = COMFY_DIR
print(f'  Stored AEI_COMFY_PROC / AEI_COMFY_URL in builtins.')
print(f'  ComfyUI ready on {COMFY_URL}. Try the GUI at that URL.')



In [ ]:
#@title STEP 4 — (Optional) Gradio UI for LTX-2.5

"""
Optional UI step. Same pattern as MiniMax-H3 notebook — exposes
prompt + canvas + steps + an audio override via gradio.
"""
# If you don't want the UI, skip this cell and use STEP 6 / STEP 7
# to submit workflows via REST directly (they call _build_workflow()
# the same way STEP 4 does).
print('STEP 4 is optional for LTX-2.5; most users will use STEP 6/7 instead.')
print('See _build_workflow() in STEP 6 for the workflow template.')



In [ ]:
#@title STEP 5 — Keep-alive + session summary (ComfyUI status)

import time, json, urllib.request, builtins

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('LTX-2.5 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "AEI_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "AEI_COMFY_DIR", "?")}/output')
    print()
    print('  Three ways to use:')
    print('    - STEP 6 (quick test): single video via form params, polls /history')
    print('    - STEP 7 (batch): JSON scene list for production runs')
    print('  STEP 8 tails the ComfyUI log if a run stalls.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')



In [ ]:
#@title STEP 6 — Quick test (single video generation)

"""
Build a single LTX-2.5 t2v workflow and queue it via POST /prompt.
Uses the same _build_workflow pattern as STEP 4 (which is currently
a stub for LTX-2.5; STEP 6 has the full chain). Mirrors the MiniMax-H3
notebook's STEP 6 but with the LTX nodes.
"""

from pathlib import Path

import os
import sys
import time
import json
import uuid
import urllib.request
import urllib.parse
import requests
import shutil
import gc
import re
import random
import builtins
from IPython.display import display, FileLink

# Default loader-class values. The if/elif/else below REPLACES these
# based on the user's TRANSFORMER dropdown value. Defaults are set FIRST
# so even if the cell-source cache in Colab is stale and the
# if/elif/else block has been cleared, _build_workflow still has
# something to reference. _LOADER_CLASS = "UNETLoader" + the safetensors
# filename is the conservative choice - if the user picked a GGUF
# option in STEP 2, they'll get a 400 from ComfyUI (file not in
# models/diffusion_models/) which the user can fix by re-running
# STEP 1 + STEP 2 cleanly.
_LOADER_CLASS = 'UNETLoader'
_UNET_FILENAME = 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'
_TEXT_ENCODER_FILE_DEFAULT = 'gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors'


URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single source of truth: STEP 2 sets builtins._TRANSFORMER_VAR and
# builtins._QUANT_VAR from its dropdown widgets. STEPS 6 + 7 read
# them here. If Colab was restarted between STEP 2 and the current
# cell, those are gone - re-run STEP 2 to restore.
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR", None)
_QUANT = getattr(builtins, "_QUANT_VAR", None)
if _TRANSFORMER is None:
    print("  ERROR: builtins._TRANSFORMER_VAR is None - STEP 2 has not been run.\n")
    print("         Re-run STEP 2 to set the model choice, then this cell.\n")
    raise SystemExit(1)
# Gemma4 text encoder filename (from STEP 2 dropdown)
_TEXT_ENCODER_FILE = getattr(builtins, "_TEXT_ENCODER_FILE",
    "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors")

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, shallow depth of field, cinematic 35mm film grain, soft ambient sound.'  #@param {type:"string"}
IMG_PATH = ''  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
LENGTH = 97  #@param {type:"slider", min:9, max:201, step:8}
STEPS = 8  #@param {type:"slider", min:1, max:20, step:1}
CFG = 1.0  #@param {type:"slider", min:1.0, max:7.0, step:0.1}
FPS = 24  #@param {type:"slider", min:16, max:60, step:1}
RESOLUTION = '832 x 480'  #@param ["768 x 432", "832 x 480", "960 x 544", "1152 x 640", "1920 x 1080"]

if SEED <= 0:
    import random
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

# Resolutions for LTX-2.5 — must be divisible by 32 and >= 256. We use
# the trained-default 768x432 / 832x480 for short clips to keep VRAM
# low; 1920x1080 is the model's training-native resolution but needs
# ~32 GB VRAM.
_CANVASES = {
    '768 x 432':   (768, 432),
    '832 x 480':   (832, 480),
    '960 x 544':   (960, 544),
    '1152 x 640':  (1152, 640),
    '1920 x 1080': (1920, 1080),
}
_width, _height = _CANVASES[RESOLUTION]
_LENGTH = LENGTH  # frames at 24 fps
print(f'  Prompt   : {PROMPT[:60]}...')
print(f'  Canvas   : {_width} x {_height} ({_LENGTH} frames)')
print(f'  Steps    : {STEPS}')
print(f'  CFG      : {CFG}')
print(f'  Seed     : {SEED}')
print()

import uuid

def _build_workflow(prompt, width, height, duration, steps, seed, cfg,
                   fps=24, first_frame=None, last_frame=None,
                   first_frame_strength=1.0, last_frame_strength=1.0,
                   filename_prefix=''):
    """Build the LTX-2.5 t2v/i2v workflow with optional first/last frame.

    Mirrors STEP 7 _build_workflow signature exactly. See STEP 7
    docstring for the full rationale. With first_frame / last_frame
    provided, LTXVAddGuide is inserted between the empty latent and
    the sampler to condition on the supplied keyframes.
    """
    length = max(9, int(round(duration * fps)))
    length = min(length, 480)
    p = {}
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {
        'unet_name': _UNET_FILENAME,
        'weight_dtype': 'default',
    }}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}
    # Text encoder (CLIP) + dual conditioning chain. LTXVConditioning
    # doesn't encode text itself - it only sets frame_rate on the
    # pre-encoded conditioning. So we need CLIPLoader + 2x CLIPTextEncode
    # (one positive, one empty/negative) feeding into LTXVConditioning.
    # The 'ltxv' type tells ComfyUI's CLIPLoader to apply the LTXV
    # text_embedding_projection + audio_projector layers from the gemma4 file.
    # The CLIP itself is loaded on-demand when the sampler starts, not now.
    p['387'] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    # Positive prompt - encoded with the LTXV-augmented CLIP.
    p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': ['387', 0],
        'text': prompt,
    }}
    # Negative prompt - empty string is the Lightricks default for
    # distilled 8-step schedules (CFG=1 doesn't use it anyway, but
    # CFGGuider still needs both inputs wired).
    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': ['387', 0],
        'text': '',
    }}
    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': ['364', 0],
        'negative': ['373', 0],
        'frame_rate': fps,
    }}
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    latent_upstream = ['4', 0]
    cond_upstream_pos = ['10', 0]
    cond_upstream_neg = ['10', 1]  # LTXVConditioning negative output slot
    _next_id = 100
    def _add_node(spec):
        nonlocal _next_id
        nid = str(_next_id)
        _next_id += 1
        p[nid] = spec
        return nid
    chains = []
    if first_frame and not last_frame:
        chains.append((first_frame, 0, first_frame_strength))
    elif last_frame and not first_frame:
        chains.append((last_frame, -1, last_frame_strength))
    elif first_frame and last_frame:
        chains.append((first_frame, 0, first_frame_strength))
        chains.append((last_frame, -1, last_frame_strength))
    for image_var, frame_idx, strength in chains:
        nid_load = _add_node({'class_type': 'LoadImage', 'inputs': {'image': image_var}})
        nid_add = _add_node({'class_type': 'LTXVAddGuide', 'inputs': {
            'positive': cond_upstream_pos,
            'negative': cond_upstream_neg,
            'vae': ['27', 0],
            'latent': latent_upstream,
            'image': [nid_load, 0],
            'frame_idx': frame_idx,
            'strength': strength,
        }})
        cond_upstream_pos = [nid_add, 0]
        cond_upstream_neg = [nid_add, 1]
        latent_upstream = [nid_add, 2]
    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': cond_upstream_pos,
        'negative': cond_upstream_neg,
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}}
    # Sigmas: ManualSigmas takes a single STRING field with comma-
    # separated floats. For the 8-step distilled schedule the model
# was post-trained for, the canonical sigmas are the 9-sigma curve
# from realrebelai/LTX-2.5_GGUFs/T2V_I2V_Single_Stage_Distilled.json:
#   "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
# We pass the user's `steps` count as steps but the actual sigma
# schedule is fixed - this isn't a 1:1 mapping. For non-8 step
# counts, we'd want a different schedule; for the distilled recipe
# always use these 9 sigmas regardless of `steps`.
    _SIGMAS_DISTILLED_8 = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    p["9"] = {"class_type": "ManualSigmas", "inputs": {
        "sigmas": _SIGMAS_DISTILLED_8,
    }}
    p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
        'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
        'sigmas': ['9', 0], 'latent_image': latent_upstream,
    }}
    p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {
        'samples': ['11', 0], 'audio_vae': ['28', 0],
    }}
    p['14'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
        'samples': ['11', 0], 'vae': ['27', 0],
        'tile_size': 512, 'overlap': 64,
        'temporal_size': 64, 'temporal_overlap': 8,
    }}
    p['40'] = {'class_type': 'CreateVideo', 'inputs': {
        'images': ['14', 0], 'audio': ['13', 0], 'fps': fps,
    }}
    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix,
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}


_slug = re.sub(r'[^a-zA-Z0-9_\-]+', '-', PROMPT).strip('-')[:40]
import re as _re
_slug = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', PROMPT).strip('-')[:40]
ts = int(time.time())
_prefix = (f'video/LTX_quicktest_{_slug}_{_width}x{_height}_f{_LENGTH}'
           f'_s{STEPS}_seed{SEED}_{ts}')

wf = _build_workflow(PROMPT, _width, _height,
                      duration=max(1, int(round(_LENGTH / max(FPS, 1)))),
                      steps=STEPS, seed=SEED, cfg=CFG,
                      fps=FPS, filename_prefix=_prefix)
nodes = wf['prompt']

# SaveVideo needs a filename_prefix field
nodes["92"] = {"class_type": "SaveVideo", "inputs": {
    "video": ["40", 0],
    "filename_prefix": _prefix,
    "format": "auto",
    "codec": "auto",
}}

t0 = time.time()
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued: {prompt_id[:8]} ... polling /history')

# Poll for completion.
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {elapsed:.0f}s ({(elapsed/STEPS):.1f}s/step est).')
            break
        if entry.get('status', {}).get('error'):
            raise gr.Error('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
        time.sleep(3)

# Resolve output file (ComfyUI strips the directory prefix from the
# filename reported in /history).
import re
paths = []
for node_out in entry.get('outputs', {}).values():
    for out in node_out.get('videos', []):
        fn = out.get('filename')
        if not fn:
            continue
        candidates = [OUT_DIR / fn]
        if '/' not in fn:
            candidates.append(OUT_DIR / 'video' / fn)
        local = None
        for c in candidates:
            if c.exists():
                local = c
                break
        if local is None:
            local = candidates[0]
        paths.append(local)
print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4','.webm','.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
display(FileLink(video_path))



In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list

"""
Advanced batch processor. Reads a JSON file containing a list of scenes, each with
its own prompt, optional keyframe images, canvas, duration, steps, and seed. The
ComfyUI subprocess is shared across scenes; each scene is a separate workflow.

JSON format (a list of objects):
```json
[
  {
    "prompt": "A red fox in a snowy pine forest at dawn",
    "first_frame": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_frame":  "/content/drive/MyDrive/keyframes/fox_end.png",
    "canvas": "832 x 480 - 16:9",
    "duration": 5,
    "steps": 8,
    "fps": 24,
    "seed": 42,
    "first_frame_strength": 1.0,
    "last_frame_strength": 1.0
  }
]
```

Fields (only `prompt` is required; rest override the DEFAULTS widgets below):
  - prompt:               (required) text description of the scene
  - first_frame:          (optional) path to first-frame image (I2V start)
  - last_frame:           (optional) path to last-frame image (I2V end; negative frame_idx)
  - canvas:               (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:             (optional, default DEFAULT_DURATION) seconds (1-20)
  - steps:                (optional, default DEFAULT_STEPS) inference steps (1-100)
  - fps:                  (optional, default DEFAULT_FPS) frames per second (16-60)
  - seed:                 (optional, default 0) 0 or negative = random
  - first_frame_strength: (optional, default 1.0) conditioning strength for first frame
  - last_frame_strength:  (optional, default 1.0) conditioning strength for last frame

Frame count = max(9, duration * fps); LTX-2.5 has a hard minimum of 9 frames.
We cap at 480 frames (20s at DEFAULT_FPS=24) to keep VRAM headroom.

If `SKIP_EXISTING` is True, runs resume by checking for existing files at
the scene index. A batch_log.jsonl file is also written for partial-resume
scenarios where the user manually deleted outputs.

If the JSON file does not exist yet, this cell writes a starter template.
"""

from pathlib import Path

import os
import sys
import time
import json
import uuid
import urllib.request
import urllib.parse
import requests
import shutil
import gc
import re
import random
import builtins
from IPython.display import display, FileLink

# Default loader-class values. The if/elif/else below REPLACES these
# based on the user's TRANSFORMER dropdown value. Defaults are set FIRST
# so even if the cell-source cache in Colab is stale and the
# if/elif/else block has been cleared, _build_workflow still has
# something to reference. _LOADER_CLASS = "UNETLoader" + the safetensors
# filename is the conservative choice - if the user picked a GGUF
# option in STEP 2, they'll get a 400 from ComfyUI (file not in
# models/diffusion_models/) which the user can fix by re-running
# STEP 1 + STEP 2 cleanly.
_LOADER_CLASS = 'UNETLoader'
_UNET_FILENAME = 'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'
_TEXT_ENCODER_FILE_DEFAULT = 'gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors'


URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single source of truth: STEP 2 sets builtins._TRANSFORMER_VAR and
# builtins._QUANT_VAR from its dropdown widgets. STEPS 6 + 7 read
# them here. If Colab was restarted between STEP 2 and the current
# cell, those are gone - re-run STEP 2 to restore.
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR", None)
_QUANT = getattr(builtins, "_QUANT_VAR", None)
if _TRANSFORMER is None:
    print("  ERROR: builtins._TRANSFORMER_VAR is None - STEP 2 has not been run.\n")
    print("         Re-run STEP 2 to set the model choice, then this cell.\n")
    raise SystemExit(1)
# Gemma4 text encoder filename (from STEP 2 dropdown)
_TEXT_ENCODER_FILE = getattr(builtins, "_TEXT_ENCODER_FILE",
    "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors")


def _fmt(seconds):
    """Format seconds as H:MM:SS or MM:SS.

    Used by all the elapsed-time status prints so they read identically
    whether the clip took 90s (the H3 case) or 1800s (a slow L4 first run).
    The user doesn't have to do unit math.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/batch_scenes.json'  #@param {type:"string"}
# === DEFAULTS: per-scene JSON keys override these. Single source of truth
# for batch-wide defaults in the form widgets, with per-scene overrides
# in the JSON.
DEFAULT_DURATION = 5  #@param {type:"slider", min:1, max:20, step:1}
DEFAULT_STEPS    = 8  #@param {type:"slider", min:1, max:100, step:1}
DEFAULT_FPS      = 24  #@param {type:"slider", min:16, max:60, step:1}
DEFAULT_FIRST_FRAME_STRENGTH = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
DEFAULT_LAST_FRAME_STRENGTH  = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
# DEFAULT_CANVAS dropdown. Label syntax is just a display nicety - the
# CANVASES dict below is the source of truth for (width, height).
DEFAULT_CANVAS = "832 x 480 - 16:9"  #@param ["1152 x 640 - 16:9", "960 x 544 - 16:9", "832 x 480 - 16:9", "704 x 400 - 16:9", "576 x 320 - 16:9", "640 x 1152 - 9:16", "544 x 960 - 9:16", "480 x 832 - 9:16", "400 x 704 - 9:16", "320 x 576 - 9:16", "768 x 768 - 1:1", "576 x 576 - 1:1", "768 x 576 - 4:3", "576 x 768 - 3:4", "1344 x 544 - ~21:9", "1024 x 416 - ~21:9"] {"allow-input": true}
SKIP_EXISTING = True  #@param {type:"boolean"}
RESUME_FROM_LOG = True  #@param {type:"boolean"}

CANVASES = {
    # 16:9 landscape
    "1152 x 640 - 16:9":     (1152, 640),
    "960 x 544 - 16:9":      (960, 544),
    "832 x 480 - 16:9":      (832, 480),
    "704 x 400 - 16:9":      (704, 400),
    "576 x 320 - 16:9":      (576, 320),
    # 9:16 portrait
    "640 x 1152 - 9:16":     (640, 1152),
    "544 x 960 - 9:16":      (544, 960),
    "480 x 832 - 9:16":      (480, 832),
    "400 x 704 - 9:16":      (400, 704),
    "320 x 576 - 9:16":      (320, 576),
    # 1:1 square
    "768 x 768 - 1:1":       (768, 768),
    "576 x 576 - 1:1":       (576, 576),
    # 4:3 / 3:4
    "768 x 576 - 4:3":       (768, 576),
    "576 x 768 - 3:4":       (576, 768),
    # Cinematic wide (~21:9)
    "1344 x 544 - ~21:9":    (1344, 544),
    "1024 x 416 - ~21:9":    (1024, 416),
}

json_path = Path(BATCH_JSON_PATH)
if not json_path.exists():
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {"prompt": "A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 42},
        {"prompt": "A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 7},
        {"prompt": "A cellist playing a slow melody in an empty concert hall, warm stage lighting, distant applause at the end.",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION,
         "steps": DEFAULT_STEPS, "fps": DEFAULT_FPS, "seed": 99},
    ]
    json_path.write_text(json.dumps(template, indent=2))
    print(f'Created batch {json_path} with {len(template)} starter scenes.')
    print('Edit it (add first_frame / last_frame / adjust defaults), then re-run STEP 7.')

with json_path.open() as f:
    scenes = json.load(f)
if not isinstance(scenes, list):
    raise SystemExit(f'Expected a JSON list, got {type(scenes).__name__}')
print(f'Loaded {len(scenes)} scene(s) from {json_path}')

# Resume log - append-only JSONL of completed scenes. Used to skip
# already-rendered scenes when RESUME_FROM_LOG is True.
_log_path = json_path.parent / (json_path.stem + '.log.jsonl')
_completed = set()
if RESUME_FROM_LOG and _log_path.exists():
    with _log_path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if rec.get('status') == 'ok':
                    _completed.add(rec['index'])
            except (json.JSONDecodeError, KeyError):
                continue
    if _completed:
        print(f'Resume log: {len(_completed)} scene(s) already completed (will skip).')


# GGUF loader detection: TRANSFORMER option containing "GGUF" means
# the city96/ComfyUI-GGUF unet path. CCTechUnetLoader is a
# separate custom node installed via ComfyUI Manager.
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]
print(f"  Loader       : {_LOADER_CLASS} (on {_TRANSFORMER})")
print(f"  Transformer  : {_UNET_FILENAME}")

def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug."""
    s = re.sub(r'[^a-zA-Z0-9_-]+', '-', s).strip('-')
    s = re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

def _frames_for_duration(duration_s, fps):
    """Convert seconds + fps to a frame count valid for LTX-2.5 (min 9, max 480)."""
    n = max(9, int(round(duration_s * fps)))
    return min(n, 480)

def _build_workflow(prompt, width, height, duration, steps, seed, cfg,
                   fps=24, first_frame=None, last_frame=None,
                   first_frame_strength=1.0, last_frame_strength=1.0,
                   filename_prefix=''):
    """Build the LTX-2.5 t2v/i2v workflow with optional first/last frame conditioning.

    When first_frame or last_frame is provided, the workflow inserts:
      LoadImage -> VAE.encode (inside) -> LTXVAddGuide
    between the conditioning chain and the sampler. LTXVAddGuide chains
    sequentially - first frame attaches at frame_idx=0, last frame at
    frame_idx=-1 (negative = counted from end, AddGuide resolves this).
    Empty latent is the input; the chained guide output becomes the
    new latent input to SamplerCustomAdvanced.
    """
    length = _frames_for_duration(duration, fps)
    p = {}
    # Model + VAE loaders
    p['6'] = {'class_type': _LOADER_CLASS, 'inputs': {
        'unet_name': _UNET_FILENAME,
        'weight_dtype': 'default',
    }}
    p['27'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-video-vae-conv-bf16.safetensors'}}
    p['28'] = {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ltx-2.5-audio-vae-bf16.safetensors'}}
    # Text conditioning
    # Text encoder (CLIP) + dual conditioning chain. LTXVConditioning
    # doesn't encode text itself - it only sets frame_rate on the
    # pre-encoded conditioning. So we need CLIPLoader + 2x CLIPTextEncode
    # (one positive, one empty/negative) feeding into LTXVConditioning.
    # The 'ltxv' type tells ComfyUI's CLIPLoader to apply the LTXV
    # text_embedding_projection + audio_projector layers from the gemma4 file.
    # The CLIP itself is loaded on-demand when the sampler starts, not now.
    p['387'] = {'class_type': 'CLIPLoader', 'inputs': {
        'clip_name': _TEXT_ENCODER_FILE,
        'type': 'ltxv',
        'device': 'default',
    }}
    # Positive prompt - encoded with the LTXV-augmented CLIP.
    p['364'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': ['387', 0],
        'text': prompt,
    }}
    # Negative prompt - empty string is the Lightricks default for
    # distilled 8-step schedules (CFG=1 doesn't use it anyway, but
    # CFGGuider still needs both inputs wired).
    p['373'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': ['387', 0],
        'text': '',
    }}
    p['10'] = {'class_type': 'LTXVConditioning', 'inputs': {
        'positive': ['364', 0],
        'negative': ['373', 0],
        'frame_rate': fps,
    }}
    # Empty latent
    p['4'] = {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {
        'width': width, 'height': height, 'length': length, 'batch_size': 1,
    }}
    # Optional first/last frame conditioning via LTXVAddGuide
    latent_upstream = ['4', 0]
    cond_upstream_pos = ['10', 0]
    cond_upstream_neg = ['10', 1]  # LTXVConditioning negative output slot
    _next_id = 100
    def _add_node(spec):
        nonlocal _next_id
        nid = str(_next_id)
        _next_id += 1
        p[nid] = spec
        return nid
    chains = []
    if first_frame and not last_frame:
        chains.append((first_frame, 0, first_frame_strength))
    elif last_frame and not first_frame:
        chains.append((last_frame, -1, last_frame_strength))
    elif first_frame and last_frame:
        chains.append((first_frame, 0, first_frame_strength))
        chains.append((last_frame, -1, last_frame_strength))
    for image_var, frame_idx, strength in chains:
        nid_load = _add_node({'class_type': 'LoadImage', 'inputs': {'image': image_var}})
        nid_add = _add_node({'class_type': 'LTXVAddGuide', 'inputs': {
            'positive': cond_upstream_pos,
            'negative': cond_upstream_neg,
            'vae': ['27', 0],
            'latent': latent_upstream,
            'image': [nid_load, 0],
            'frame_idx': frame_idx,
            'strength': strength,
        }})
        # LTXVAddGuide: [0]=positive [1]=negative [2]=latent
        cond_upstream_pos = [nid_add, 0]
        cond_upstream_neg = [nid_add, 1]
        latent_upstream = [nid_add, 2]
    # Sampler chain
    p['5'] = {'class_type': 'CFGGuider', 'inputs': {
        'model': ['6', 0],
        'positive': cond_upstream_pos,
        'negative': cond_upstream_neg,
        'cfg': cfg,
    }}
    p['7'] = {'class_type': 'RandomNoise', 'inputs': {'noise_seed': seed}}
    p['8'] = {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}}
    # Sigmas: ManualSigmas takes a single STRING field with comma-
    # separated floats. For the 8-step distilled schedule the model
# was post-trained for, the canonical sigmas are the 9-sigma curve
# from realrebelai/LTX-2.5_GGUFs/T2V_I2V_Single_Stage_Distilled.json:
#   "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
# We pass the user's `steps` count as steps but the actual sigma
# schedule is fixed - this isn't a 1:1 mapping. For non-8 step
# counts, we'd want a different schedule; for the distilled recipe
# always use these 9 sigmas regardless of `steps`.
    _SIGMAS_DISTILLED_8 = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    p["9"] = {"class_type": "ManualSigmas", "inputs": {
        "sigmas": _SIGMAS_DISTILLED_8,
    }}
    p['11'] = {'class_type': 'SamplerCustomAdvanced', 'inputs': {
        'noise': ['7', 0], 'guider': ['5', 0], 'sampler': ['8', 0],
        'sigmas': ['9', 0], 'latent_image': latent_upstream,
    }}
    # Decode + composite + save
    p['13'] = {'class_type': 'LTXVAudioVAEDecode', 'inputs': {
        'samples': ['11', 0], 'audio_vae': ['28', 0],
    }}
    p['14'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
        'samples': ['11', 0], 'vae': ['27', 0],
        'tile_size': 512, 'overlap': 64,
        'temporal_size': 64, 'temporal_overlap': 8,
    }}
    p['40'] = {'class_type': 'CreateVideo', 'inputs': {
        'images': ['14', 0], 'audio': ['13', 0], 'fps': fps,
    }}
    p['92'] = {'class_type': 'SaveVideo', 'inputs': {
        'video': ['40', 0], 'filename_prefix': filename_prefix,
        'format': 'auto', 'codec': 'auto',
    }}
    return {'prompt': p}

def _upload_image(path):
    """Upload a local image to ComfyUI server (returns the server-side filename)."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    with p.open('rb') as f:
        mime = 'image/png' if p.suffix.lower() == '.png' else 'image/jpeg'
        files = {'image': (p.name, f, mime)}
        data = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(f'{URL}/upload/image', files=files, data=data, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f'upload failed: {r.status_code} {r.text[:300]}')
    return r.json()['name']

results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    if i in _completed:
        print(f'  [{i+1}/{len(scenes)}] SKIP (resume log)')
        results.append(None)
        continue
    prompt = sc.get('prompt', '').strip()
    if not prompt:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty prompt')
        results.append(None)
        continue
    canvas_label = sc.get('canvas', DEFAULT_CANVAS)
    w, h = CANVASES.get(canvas_label, (832, 480))
    duration = int(sc.get('duration', DEFAULT_DURATION))
    fps      = int(sc.get('fps',      DEFAULT_FPS))
    steps    = int(sc.get('steps',    DEFAULT_STEPS))
    seed     = int(sc.get('seed',     0))
    if seed <= 0:
        seed = random.randint(1, 2**31 - 1)
    ff_strength = float(sc.get('first_frame_strength', DEFAULT_FIRST_FRAME_STRENGTH))
    lf_strength = float(sc.get('last_frame_strength',  DEFAULT_LAST_FRAME_STRENGTH))
    ff = sc.get('first_frame', '').strip() or None
    lf = sc.get('last_frame',  '').strip() or None
    length = _frames_for_duration(duration, fps)
    scene_slug = _slug(prompt, maxlen=30)
    scene_ts = int(time.time())
    scene_prefix = (f'video/LTX_batch_{i:03d}_{scene_slug}_{w}x{h}_d{duration}s_f{length}'
                    f'_s{steps}_seed{seed}_{scene_ts}')
    # SKIP_EXISTING checks the output filesystem
    if SKIP_EXISTING:
        existing = list(OUT_DIR.glob(f'LTX_batch_{i:03d}_*'))
        if existing:
            print(f'  [{i+1}/{len(scenes)}] SKIP (already exists: {existing[0].name})')
            with _log_path.open('a') as f:
                f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(existing[0])}) + chr(10))
            _completed.add(i)
            results.append(str(existing[0]))
            continue
    # Upload first/last frame images if provided
    try:
        ff_name = _upload_image(ff) if ff else None
        lf_name = _upload_image(lf) if lf else None
    except (FileNotFoundError, RuntimeError) as e:
        print(f'  [{i+1}/{len(scenes)}] FAIL: image upload: {e}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'image_error', 'error': str(e)}) + chr(10))
        continue
    print(f'\n  [{i+1}/{len(scenes)}] {w}x{h} d{duration}s f{length} {steps}steps fps={fps} seed={seed}')
    print(f'    {prompt[:80]}')
    if ff:
        print(f'    first_frame: {ff} (strength={ff_strength})')
    if lf:
        print(f'    last_frame:  {lf} (strength={lf_strength})')
    cfg = 1.0
    wf = _build_workflow(prompt=prompt, width=w, height=h, duration=duration,
                          steps=steps, seed=seed, cfg=cfg, fps=fps,
                          first_frame=ff_name, last_frame=lf_name,
                          first_frame_strength=ff_strength, last_frame_strength=lf_strength,
                          filename_prefix=scene_prefix)
    nodes = wf['prompt']
    t0 = time.time()
    r = requests.post(f'{URL}/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'submit_error', 'code': r.status_code}) + chr(10))
        continue
    pid = r.json()['prompt_id']
    print(f'    Queued as {pid[:8]} - polling /history for completion ...')
    last_report = 0
    log_path = Path(getattr(builtins, 'AEI_COMFY_LOG', '/content/drive/MyDrive/AEI_ComfyUI/comfyui.log'))
    last_log_size = 0
    if log_path.exists():
        last_log_size = log_path.stat().st_size
    completed_clean = False
    while True:
        try:
            h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        except (requests.exceptions.RequestException, ValueError):
            time.sleep(5)
            continue
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                elapsed = time.time() - t0
                print(f'    Done in {_fmt(elapsed)} ({(elapsed / steps):.1f}s/step).')
                completed_clean = True
                break
            if entry.get('status', {}).get('error'):
                err = entry['status'].get('messages', [])
                print(f'    FAIL: {json.dumps(err)[:300]}')
                break
        elapsed = time.time() - t0
        # Pulse print: log changes + 5-min heartbeat
        new_lines = []
        if log_path.exists():
            cur_size = log_path.stat().st_size
            if cur_size > last_log_size:
                with log_path.open('rb') as f:
                    f.seek(last_log_size)
                    new_lines = f.read().decode('utf-8', errors='replace').splitlines()
                last_log_size = cur_size
        tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
        if tail or time.time() - last_report > 300:
            last_report = time.time()
            print(f'    ... {_fmt(elapsed)} elapsed' + (f'  log: {tail}' if tail else ''), flush=True)
        time.sleep(5)
    if not completed_clean:
        print('    FAIL: polling exited without completion')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'poll_failed'}) + chr(10))
        gc.collect()
        continue
    # Find SaveVideo output and pull locally
    elapsed = time.time() - t0
    local_path = None
    for node_id, node_out in entry.get('outputs', {}).items():
        for v in node_out.get('videos', []):
            fn = v['filename']
            local_candidates = [OUT_DIR / fn]
            if '/' not in fn:
                local_candidates.append(OUT_DIR / 'video' / fn)
            for _wait in range(7):
                for c in local_candidates:
                    if c.exists():
                        local_path = c
                        break
                if local_path is not None:
                    break
                if _wait == 3 and local_path is None:
                    for hit in OUT_DIR.glob(f'*/{fn}'):
                        local_path = hit
                        break
                time.sleep(2)
            if local_path is None:
                local_path = local_candidates[0]
                url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type=output'
                try:
                    with urllib.request.urlopen(url, timeout=120) as r2, open(local_path, 'wb') as fp:
                        shutil.copyfileobj(r2, fp)
                except urllib.error.HTTPError as e:
                    print(f'    WARN: server /view returned {e.code}; falling back to local-only')
            break
        if local_path is not None:
            break
    if local_path is not None:
        print(f'    {local_path.name}  ({_fmt(elapsed)})')
        results.append(str(local_path))
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(local_path),
                                'duration_s': elapsed}) + chr(10))
    else:
        print('    FAIL: no video output from SaveVideo')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'no_output'}) + chr(10))
    gc.collect()

total = time.time() - total_start
ok_count = sum(1 for r in results if r)
print(f'\nBatch complete: {ok_count}/{len(results)} clips, total {_fmt(total)} ({total / max(ok_count, 1):.0f}s/clip)')
for p in [r for r in results if r]:
    print(f'  {p}')


In [ ]:
#@title STEP 8 — Tail ComfyUI log (debugging aid)

import time
from pathlib import Path

LOG = Path(getattr(__import__('builtins'), 'AEI_COMFY_LOG',
                     '/content/drive/MyDrive/AEI_ComfyUI/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:10, max:500, step:10}

if LOG.exists():
    lines = LOG.read_text(errors='replace').splitlines()
    print(f'  Last {min(TAIL_LINES, len(lines))} of {len(lines)} log lines from {LOG}:')
    print()
    for line in lines[-TAIL_LINES:]:
        print(f'    {line}')
else:
    print(f'  No log file at {LOG}')



In [ ]:
#@title STEP 8.5 — Smoke-test the workflow wiring

"""
Run a minimal t2v workflow at LENGTH=9, STEPS=2 with a tiny canvas.
Verifies the UNETLoader + VAELoader + LTXVConditioning chain wires
up correctly. Should complete in ~30-60 s on L4 (dominated by
model load time, not the denoise itself).
"""
import json, time, uuid, requests, builtins
from pathlib import Path
URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'

w, h, length, steps, cfg = 256, 256, 9, 2, 1.0

# Same loader detection as STEP 6 (duplicated for self-containment).
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR",
                                  'Lightricks/LTX-2.5 (distilled int8-convrot)')
_QUANT = getattr(builtins, "_QUANT_VAR", "Q4_K_M")
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]
seed = 12345

# Minimal workflow (matches Lightricks example_workflows/2.5/).
nodes = {
    "6": {"class_type": _LOADER_CLASS, "inputs": {
        "unet_name": _UNET_FILENAME,
        "weight_dtype": "default",
    }},
    "27": {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-video-vae-conv-bf16.safetensors",
    }},
    "28": {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-audio-vae-bf16.safetensors",
    }},
    "10": {"class_type": "LTXVConditioning", "inputs": {
        "prompt": "A fluffy white cloud drifting across a blue sky.",
    }},
    "4": {"class_type": "EmptyLTXVLatentVideo", "inputs": {
        "width": w, "height": h, "length": length, "batch_size": 1,
    }},
    "5": {"class_type": "CFGGuider", "inputs": {
        "model": ["6", 0], "positive": ["10", 0], "negative": ["10", 0], "cfg": cfg,
    }},
    "7": {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}},
    "8": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
    "9": {"class_type": "ManualSigmas", "inputs": {
        "steps": steps, "denoise_min": 0.0, "denoise_max": 1.0,
        "sigma_min": 1.0, "sigma_max": 0.0,
    }},
    "11": {"class_type": "SamplerCustomAdvanced", "inputs": {
        "noise": ["7", 0], "guider": ["5", 0], "sampler": ["8", 0],
        "sigmas": ["9", 0], "latent_image": ["4", 0],
    }},
    "13": {"class_type": "LTXVAudioVAEDecode", "inputs": {
        "samples": ["11", 0], "audio_vae": ["28", 0],
    }},
    "14": {"class_type": "VAEDecodeTiled", "inputs": {
        "samples": ["11", 0], "vae": ["27", 0], "tile_size": 256, "overlap": 32,
    }},
    "40": {"class_type": "CreateVideo", "inputs": {
        "images": ["14", 0], "audio": ["13", 0], "fps": 24,
    }},
    "92": {"class_type": "SaveVideo", "inputs": {
        "video": ["40", 0],
        "filename_prefix": f"smoketest/LTX_{w}x{h}_s{steps}_{int(time.time())}",
        "format": "auto", "codec": "auto",
    }},
}

print(f'  POST /prompt: {w}x{h}, length={length}, {steps} steps ...')
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'Workflow rejected: {r.status_code}\n{r.text[:1000]}')
pid = r.json()['prompt_id']
print(f'  Queued as {pid[:8]}; polling /history for completion ...')
t0 = time.time()
while True:
    h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
    if pid in h:
        entry = h[pid]
        if entry.get('status', {}).get('completed'):
            print(f'  Done in {time.time() - t0:.0f}s.')
            for node_out in entry.get('outputs', {}).values():
                for out in node_out.get('videos', []):
                    fn = out.get('filename')
                    if fn:
                        print(f'  Output: {OUT_DIR / fn}')
            break
        if entry.get('status', {}).get('error'):
            print('  FAIL:')
            for msg in entry['status'].get('messages', []):
                print(f'    {msg}')
            break
    time.sleep(3)

